# Menzili Properties Data Transformation

This notebook transforms property data from JSON format to a structured CSV file. Let's break down what we're doing:

1. We'll load property listings from a JSON file
2. Transform amenities into binary columns (0 = has feature, 1 = doesn't have feature)
3. Clean and standardize numeric data (prices, surfaces, rooms)
4. Save everything in a clean CSV format

In [ ]:
import pandas as pd
import json
import os
from typing import Dict, List, Any

## Data Loading Function
First, we need to load our JSON data. This function opens the JSON file and returns its contents as a Python list of dictionaries.

In [ ]:
def load_json_data(file_path: str) -> List[Dict[str, Any]]:
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)

def get_unique_amenities(df: pd.DataFrame) -> set:
    all_amenities = set()
    for options in df['options']:
        if isinstance(options, list):
            all_amenities.update(options)
    return all_amenities

## Amenities Processing
These functions handle the amenities (options) in our data:
1. First function finds all unique amenities across all properties
2. Second function creates binary columns for each amenity (0 = has feature, 1 = doesn't have it)
## Numeric Data Cleaning
This function handles all the numeric data cleaning:
1. Cleans price values (removes 'DT', spaces, handles commas)
2. Processes surface areas (handles different formats like 'm²', 'm2', and summed areas like '150+100')
3. Converts other numeric fields (rooms, bathrooms, etc.)

In [ ]:
def create_amenity_columns(df: pd.DataFrame, amenities: set) -> pd.DataFrame:
    for amenity in amenities:
        col_name = f'amenity_{amenity.lower().replace(" / ", "_").replace(" ", "_")}'
        df[col_name] = df['options'].apply(
            lambda x: 0 if isinstance(x, list) and amenity in x else 1
        )
    return df


## Numeric Data Cleaning
This function handles all the numeric data cleaning:
1. Cleans price values (removes 'DT', spaces, handles commas)
2. Processes surface areas (handles different formats like 'm²', 'm2', and summed areas like '150+100')
3. Converts other numeric fields (rooms, bathrooms, etc.)

In [ ]:
def clean_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:
    # Price cleaning remains the same
    df['price'] = df['price'].replace('Contactez l\'annonceur', 'NaN')
    df['price'] = df['price'].str.replace(' DT', '')
    df['price'] = df['price'].str.replace(' ', '').str.replace(',', '.')
    df['price'] = pd.to_numeric(df['price'], errors='coerce')
    
    # Surface cleaning remains the same
    for col in ['property_details_surface_habitable', 'property_details_surface_terrain']:
        df[col] = df[col].fillna('0 m²')
        df[col] = df[col].str.replace(' m²', '').str.replace(' m', '')
        df[col] = df[col].str.replace('m²', '').str.replace('m2', '')
        df[col] = df[col].apply(lambda x: str(sum(float(num) for num in str(x).split('+')) 
                                            if isinstance(x, str) and '+' in x 
                                            else x))
        df[col] = df[col].str.replace(' ', '').str.replace(',', '.').astype(float)
    
    # Modified handling for columns that might have "10+" values
    plus_cols = ['property_details_chambres', 'property_details_salle_de_bain', 
                'property_details_pieces']
    
    for col in plus_cols:
        # Keep these columns as strings to preserve the "+" notation
        # First ensure the column is treated as string
        df[col] = df[col].astype(str)
        # Replace NaN values with empty string
        df[col] = df[col].replace('nan', '')
    
    # Year of construction should still be numeric if possible
    df['property_details_annee_construction'] = pd.to_numeric(
        df['property_details_annee_construction'], errors='coerce')
    
    return df

## Main Processing
Now let's put everything together and process our data:

In [ ]:
# Directory structure verification
os.makedirs('c:/Users/MSI/Desktop/House Sale Price Prediction/JSON', exist_ok=True)
os.makedirs('c:/Users/MSI/Desktop/House Sale Price Prediction/CSV', exist_ok=True)

# File path configuration
input_file = 'c:/Users/MSI/Desktop/House Sale Price Prediction/JSON/menzili_properties_data_app.json'
output_file = 'c:/Users/MSI/Desktop/House Sale Price Prediction/CSV/menzili_properties.csv'

# Data acquisition and processing
data = load_json_data(input_file)
df = pd.json_normalize(data, sep='_', max_level=1)

# Feature processing
amenities = get_unique_amenities(df)
df = create_amenity_columns(df, amenities)
df = clean_numeric_columns(df)

# Column structure optimization
column_order = [
    'title', 'premium', 'price', 'property_type',
    'property_details_chambres', 'property_details_salle_de_bain',
    'property_details_pieces', 'property_details_surface_habitable',
    'property_details_surface_terrain', 'property_details_annee_construction',
    'location', 'neighborhood', 'city',
    'description', 'link'
]

# Amenity column integration
amenity_columns = [col for col in df.columns if col.startswith('amenity_')]
column_order.extend(amenity_columns)

# Final data structure implementation
df = df[column_order]

# Export protocol execution
df.to_csv(output_file, index=False)

# Output verification
print("Data transformation completed successfully.")
df.head()

Data transformation completed successfully.


,title,premium,price,property_type,property_details_chambres,property_details_salle_de_bain,property_details_pieces,property_details_surface_habitable,property_details_surface_terrain,property_details_annee_construction,...,amenity_interphone,amenity_terrasses,amenity_chauffage_électriques,amenity_ascenseur,amenity_système_alarme,amenity_accès_internet,amenity_vue_mer,amenity_meublé,amenity_chauffage_centrale,amenity_cuisine_équipé
0,A vendre Appartement au lac 2,Premium,1250000.0,Appartement,3,None,4,178.0,178.0,NaN,...,1,1,1,1,1,1,1,0,1,1
1,Appartement S3 à vendre à la Marsa,Premium,360000.0,Appartement,3,None,4,110.0,110.0,NaN,...,1,1,1,1,1,1,1,1,1,1
2,Appartement S+1 à La Marsa MAV1613,Premium,320000.0,Appartement,1,1,None,53.0,0.0,NaN,...,1,1,1,1,1,1,1,1,1,1
3,À VENDRE appartement à la SOUKRA,Standard,422000.0,Appartement,2,2,3,125.0,125.0,2022.0,...,0,0,0,0,1,0,1,1,0,0
4,Un appartement S+1 tout neuf à vendre à AFH Mr...,Standard,245000.0,Appartement,1,1,2,74.0,0.0,2024.0,...,0,0,1,1,0,0,1,1,0,0
